# Un model de limbaj local cu Transformers

Laborator practic în care descărcăm de pe Hugging Face Hub un model de limbaj cu ponderi deschise și îl rulăm pe propriul calculator, în Python, cu bibliotecile Transformers și PyTorch.

Notebookul conține, în ordine, codul tuturor paginilor laboratorului [Un model de limbaj local cu Transformers](https://inteligentartificiala.ro/laboratoare/model-local-transformers/prezentare/), de pe [InteligentArtificiala.ro](https://inteligentartificiala.ro). Explicațiile fiecărui pas sunt pe pagini.

În Google Colab, codul rulează mult mai repede pe o placă grafică. Dacă nu este deja aleasă, se alege din meniul **Runtime**, opțiunea **Change runtime type**, o variantă cu GPU. Celulele se rulează în ordine, de sus în jos.

In [ ]:
%pip install -q -U transformers accelerate

## [Introducere](https://inteligentartificiala.ro/laboratoare/model-local-transformers/prezentare/)

### Primul model

Cel mai scurt drum de la nimic la un răspuns este funcția `pipeline`, care ascunde toți pașii intermediari.

In [ ]:
from transformers import pipeline

# La prima rulare, fișierele modelului (aproximativ 700 MB) se descarcă de pe Hugging Face Hub.
generator = pipeline("text-generation", model="HuggingFaceTB/SmolLM2-360M-Instruct")

messages = [{"role": "user", "content": "What is the capital of France? Answer in one sentence."}]
result = generator(messages, max_new_tokens=40)
print(result[0]["generated_text"][-1]["content"])

## [Tokenizatorul, modelul și generarea](https://inteligentartificiala.ro/laboratoare/model-local-transformers/tokenizator-si-generare/)

In [ ]:
# Modelele paginilor anterioare nu mai sunt necesare, așa că le scoatem din memorie.
%reset -f
import gc

import torch

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

### Tokenizatorul

Modelul nu vede litere, ci numere. **Tokenizatorul** (_tokenizer_) taie textul în tokenuri și înlocuiește fiecare token cu ID-ul lui din vocabular. Clasele `Auto…` aleg automat implementarea potrivită pentru modelul cerut.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-360M-Instruct")

ids = tokenizer("Transformers runs language models locally.")["input_ids"]
print("ID-uri:  ", ids)
print("Tokenuri:", tokenizer.convert_ids_to_tokens(ids))
print("Înapoi:  ", tokenizer.decode(ids))
print("Mărimea vocabularului:", len(tokenizer))

### Modelul și șablonul de chat

Codul de mai jos încarcă modelul, construiește promptul unei conversații și generează un răspuns.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "HuggingFaceTB/SmolLM2-360M-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, dtype="auto", device_map="auto")
print(f"Dispozitiv: {model.device}, tip de date: {model.dtype}, {model.num_parameters():,} parametri")

messages = [
    {"role": "system", "content": "You are a concise assistant."},
    {"role": "user", "content": "Explain in two sentences what a token is."},
]
# Șablonul de chat transformă lista de mesaje în textul exact pe care l-a văzut modelul la antrenare.
print(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

inputs = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True, return_dict=True, return_tensors="pt"
).to(model.device)
output = model.generate(**inputs, max_new_tokens=100, do_sample=True, temperature=0.2, top_p=0.9)

# Rezultatul conține și promptul, așa că păstrăm doar tokenurile noi.
new_tokens = output[0, inputs["input_ids"].shape[1]:]
print(tokenizer.decode(new_tokens, skip_special_tokens=True))

### O conversație afișată pe măsura generării

Asistenții conversaționali afișează textul pe măsură ce apare, nu tot deodată, la final. În Transformers, acest lucru se face cu un `TextStreamer`. Exemplul de mai jos poartă și o conversație cu două replici.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, TextStreamer

model_id = "HuggingFaceTB/SmolLM2-360M-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, dtype="auto", device_map="auto")
# Afișează fiecare bucată de text imediat ce este generată.
streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

messages = []
for question in ["Name three planets.", "Which of them is the largest?"]:
    print(f"\n> {question}")
    messages.append({"role": "user", "content": question})
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_dict=True, return_tensors="pt"
    ).to(model.device)
    output = model.generate(**inputs, max_new_tokens=80, streamer=streamer)
    # Modelul nu ține minte conversația, așa că răspunsul lui intră în istoricul trimis data viitoare.
    answer = tokenizer.decode(output[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    messages.append({"role": "assistant", "content": answer})

## [Memorie, dispozitive și cuantizare](https://inteligentartificiala.ro/laboratoare/model-local-transformers/memorie-si-dispozitive/)

In [ ]:
# Modelele paginilor anterioare nu mai sunt necesare, așa că le scoatem din memorie.
%reset -f
import gc

import torch

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

### De câtă memorie are nevoie un model

Metoda `get_memory_footprint` spune cât ocupă ponderile unui model deja încărcat.

In [ ]:
import torch
from transformers import AutoModelForCausalLM

model_id = "HuggingFaceTB/SmolLM2-360M-Instruct"
for dtype in [torch.float32, torch.bfloat16]:
    model = AutoModelForCausalLM.from_pretrained(model_id, dtype=dtype)
    print(f"{str(dtype):15s} {model.get_memory_footprint() / 1e6:6.0f} MB")

### Cuantizarea

**Cuantizarea** (_quantization_) stochează ponderile cu mai puțini biți, de exemplu pe 4 biți în loc de 16. Modelul ocupă de câteva ori mai puțină memorie și pierde, de obicei, puțin din calitate. Cea mai simplă metodă în Transformers folosește biblioteca **bitsandbytes**, care cuantizează ponderile chiar în momentul încărcării. Pentru a o instala, rulați:

In [ ]:
%pip install -q bitsandbytes

Codul de mai jos încarcă același model pe 4 biți, în formatul NF4.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

quantization = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16
)
model = AutoModelForCausalLM.from_pretrained(
    "HuggingFaceTB/SmolLM2-360M-Instruct", quantization_config=quantization, device_map="auto"
)
print(f"4 biți (NF4): {model.get_memory_footprint() / 1e6:.0f} MB")

## [Un model de raționament local](https://inteligentartificiala.ro/laboratoare/model-local-transformers/model-de-rationament/)

In [ ]:
# Modelele paginilor anterioare nu mai sunt necesare, așa că le scoatem din memorie.
%reset -f
import gc

import torch

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

### Gândurile și răspunsul

În modul cu gândire, modelul scrie mai întâi un raționament între tokenurile `<think>` și `</think>`, apoi răspunsul. Modul se alege din șablonul de chat, cu argumentul `enable_thinking`. Codul de mai jos pune aceeași întrebare în ambele moduri și separă raționamentul de răspuns după poziția tokenului `</think>`.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "Qwen/Qwen3-1.7B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, dtype="auto", device_map="auto")
end_think = tokenizer.convert_tokens_to_ids("</think>")

messages = [{"role": "user", "content": "How many numbers from 1 to 100 are divisible by 3 or by 5?"}]

for thinking in [True, False]:
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, enable_thinking=thinking, return_dict=True, return_tensors="pt"
    ).to(model.device)
    # generation_config.json conține parametrii de eșantionare recomandați pentru modul cu gândire.
    output = model.generate(**inputs, max_new_tokens=8192)
    new_tokens = output[0, inputs["input_ids"].shape[1]:].tolist()

    # Raționamentul se termină la tokenul </think>. Fără gândire, el lipsește din răspuns.
    split = new_tokens.index(end_think) + 1 if end_think in new_tokens else 0
    reasoning = tokenizer.decode(new_tokens[:split], skip_special_tokens=True)
    answer = tokenizer.decode(new_tokens[split:], skip_special_tokens=True).strip()

    print(f"\nenable_thinking={thinking}: {split} tokenuri de gândire, {len(new_tokens) - split} în răspuns")
    print("Începutul raționamentului:", reasoning[:300])
    print("Finalul răspunsului:", answer[-120:])

### Bugetul de gândire

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "Qwen/Qwen3-1.7B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, dtype="auto", device_map="auto")


def continue_text(text, max_new_tokens, **kwargs):
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    output = model.generate(**inputs, max_new_tokens=max_new_tokens, **kwargs)
    new_tokens = output[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True), len(new_tokens)


def answer_with_budget(question, max_thinking, min_thinking=0):
    text = tokenizer.apply_chat_template(
        [{"role": "user", "content": question}], add_generation_prompt=True, enable_thinking=True, tokenize=False
    )
    used = 0
    while True:
        chunk, n = continue_text(text, max_thinking - used, stop_strings=["</think>"], tokenizer=tokenizer)
        used += n
        text += chunk.removesuffix("</think>")
        finished = chunk.endswith("</think>")
        if finished and used < min_thinking:
            text += "\nWait"  # modelul voia să se oprească prea devreme, așa că îl lăsăm să mai gândească
            continue
        if not finished:
            text += "\n\nI must give the final answer now.\n"  # bugetul s-a terminat
        break
    answer, _ = continue_text(text + "</think>\n\n", 1024)
    return used, answer.strip()


question = "How many numbers from 1 to 100 are divisible by 3 or by 5? Reply with the number only."
for max_thinking in [32, 256, 2048]:
    used, answer = answer_with_budget(question, max_thinking)
    print(f"buget maxim {max_thinking:4d}: {used:4d} tokenuri de gândire, răspuns: {answer[-40:]!r}")

used, answer = answer_with_budget(question, max_thinking=4096, min_thinking=3000)
print(f"buget minim 3000: {used} tokenuri de gândire, răspuns: {answer[-40:]!r}")